## Maham Nafees
## Roll Number 038
## Lab 12 - Medical Center QnA Bot using MiniLM + FAISS

## I Installed already

### Step 1 - Install Required Libraries

### Step 2 - Load and Preprocess the Dataset

In [ ]:
import pandas as pd
import re

# Medical center QnA dataset
data = [
    {"question": "How do I book an appointment?", "answer": "You can book an appointment by calling 042-1234567 or visiting our reception desk."},
    {"question": "What are the hospital timings?", "answer": "Our hospital is open Monday to Friday 9AM to 5PM and Saturday 10AM to 2PM."},
    {"question": "Which doctors are available in cardiology?", "answer": "Our Cardiology department has Dr. Ahmed Khan and Dr. Sara Ali."},
    {"question": "Who are the neurology doctors?", "answer": "Dr. Bilal Hassan and Dr. Fatima Noor are available in the Neurology department."},
    {"question": "What departments does the hospital have?", "answer": "We have Cardiology, Neurology, Pediatrics, Orthopedics, Emergency, and Radiology departments."},
    {"question": "Is there an emergency service?", "answer": "Yes, our Emergency Department is open 24/7. For urgent cases call 1122."},
    {"question": "Where is the hospital located?", "answer": "We are located at 123 Main Street, Lahore, near Liberty Market."},
    {"question": "What is the contact number?", "answer": "You can reach us at 042-1234567 or email info@citymedical.com."},
    {"question": "Who are the pediatrics doctors?", "answer": "Dr. Ayesha Malik and Dr. Usman Tariq are available in Pediatrics."},
    {"question": "Who are the orthopedics doctors?", "answer": "Dr. Zain Abbas and Dr. Hira Jamil handle Orthopedics cases."},
    {"question": "Can I cancel my appointment?", "answer": "Yes, you can cancel by calling us at least 24 hours before your scheduled time."},
    {"question": "Do you have radiology services?", "answer": "Yes, our Radiology department provides X-ray, MRI, and ultrasound services."},
    {"question": "What should I bring for my first visit?", "answer": "Please bring your CNIC, previous medical reports, and any prescriptions you have."},
    {"question": "Is there parking available?", "answer": "Yes, free parking is available for patients and visitors."},
    {"question": "Do you accept insurance?", "answer": "Yes, we accept most major health insurance plans. Please contact reception for details."}
]

df = pd.DataFrame(data)
print(df.shape)
df.head()

### Step 3 - Clean the Text

In [ ]:
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'[^A-Za-z\s]', '', text)
        text = text.lower()
    else:
        text = ''
    return text

# cleaning the questions for embedding
df['clean_question'] = df['question'].apply(clean_text)

print(df[['question', 'clean_question']].head())

### Step 4 - Generate Embeddings using MiniLM

In [ ]:
from sentence_transformers import SentenceTransformer

# loading the MiniLM model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

questions = df['clean_question'].tolist()
embeddings = model.encode(questions)

print("Embeddings shape:", embeddings.shape)

### Step 5 - Store Vectors in FAISS Index

In [ ]:
import faiss
import numpy as np

# converting to float32 for faiss
embeddings = np.array(embeddings).astype('float32')

d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

print("Total vectors in index:", index.ntotal)

# saving the index to disk
faiss.write_index(index, 'medical_faiss.index')
print("FAISS index saved!")

### Step 6 - Save the Dataset

In [ ]:
# saving the dataframe so flask app can load it
df.to_csv('medical_qna.csv', index=False)
print("Dataset saved to medical_qna.csv")

### Step 7 - Test the Search Function

In [ ]:
def search_answer(query, model, index, df, k=3):
    clean_query = clean_text(query)
    query_embedding = model.encode([clean_query])
    query_embedding = np.array(query_embedding).astype('float32')

    distances, indices = index.search(query_embedding, k)

    print(f"Query: {query}")
    print("-" * 40)
    for i in range(k):
        idx = indices[0][i]
        print(f"Match {i+1}: {df['question'].iloc[idx]}")
        print(f"Answer: {df['answer'].iloc[idx]}")
        print(f"Distance: {distances[0][i]:.4f}")
        print()

# testing with sample queries
search_answer("how to make appointment", model, index, df)
search_answer("heart doctor", model, index, df)
search_answer("emergency contact", model, index, df)